In [1]:
import json
from pathlib import Path
import pandas as pd
from typing import List

from model_ranking import (
    to_target_transfer_correlations,
    correlation_table,
    match_model_names,
    load_transfer_metric_results,
    dataframe_to_latex_table_styled,
)

INFO: P [MainThread] 2025-11-19 09:57:46,855 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
performance_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency/self_supervised/transfer_self_supervised_Finetuned_Def_performance_scores.json"
with open(performance_path, "r") as f:
    performance_scores = json.load(f)["performance_scores"]


# Gauss EI (finetuned)

In [3]:
base_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency/self_supervised/CMB"
selected_augmentations = ["a001-003", "a003-005", "a005-007", "a007-01", "a01-012", "a012-015", "a015-02", "a02-025", "a025-03", "a03-035", "a035-04"]

gauss_E_dfs: List[pd.DataFrame] = []

for aug in selected_augmentations:
    file_name = f"transfer_self_supervised_Gauss_{aug}_CMB_05f_05b_EI_scores.json"
    consistency_path = Path(base_path) / file_name
    results = load_transfer_metric_results(consistency_path)
    consistency_scores = results["transfer_scores"]
    targets = list(consistency_scores.keys())
    consistency_scores = match_model_names(
        performance_scores=performance_scores,
        transfer_scores=consistency_scores,
    )
    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=targets,
        transfer_metric_per_target=consistency_scores,
        performance_per_target=performance_scores,
        )
    df_gauss_EI = correlation_table(KT_scores, SP_scores, PE_scores, targets=targets, num_sig_fig=4)
    print(df_gauss_EI)
    gauss_E_dfs.append(df_gauss_EI)

Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency/self_supervised/CMB/transfer_self_supervised_Gauss_a001-003_CMB_05f_05b_EI_scores.json
                  kt  kt pval   s rho  s rho pval      pr  pr pval
Task targets                                                      
Mito EPFL     0.7091   0.0020  0.8545      0.0040  0.9268   0.0000
     Hmito    0.6364   0.0100  0.7818      0.0040  0.7134   0.0137
     Rmito    0.5636   0.0160  0.7455      0.0140  0.5212   0.1001
     VNC      0.1515   0.5435  0.2168      0.5734  0.1136   0.7252
Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency/self_supervised/CMB/transfer_self_supervised_Gauss_a003-005_CMB_05f_05b_EI_scores.json
                  kt  kt pval   s rho  s rho pval      pr  pr pval
Task targets                                                      
Mito EPFL     0.8182

In [4]:
metric_names= ['CTE-EI']*len(selected_augmentations)
names2 = ['Gauss']*len(selected_augmentations)
names3 = [ f'({aug})' for aug in selected_augmentations]

latex_table = dataframe_to_latex_table_styled(
    gauss_E_dfs, 
    metric_names, 
    names2,
    names3,
    caption="Post-UDA CTE_EI Correlation scores for Semantic Segmentation of Mitochondria across a range of Gaussian Input Perturbation Strengths. \textit{pval.} $<$ 0.05 (*), \textit{pval.} $<$ 0.01 (**).",
    label="tab:cte_ei_mitochondria_UDA_gauss"
)
print(latex_table)

\begin{table}[tb]
\centering
\scriptsize
\caption{Post-UDA CTE_EI Correlation scores for Semantic Segmentation of Mitochondria across a range of Gaussian Input Perturbation Strengths. 	extit{pval.} $<$ 0.05 (*), 	extit{pval.} $<$ 0.01 (**).}
\setlength{\tabcolsep}{3pt}
\begin{tabular}{@{}c c
   >{\columncolor{GreyTable}}c
   >{\columncolor{GreyTable}}c
   c c
   >{\columncolor{GreyTable}}c
   >{\columncolor{GreyTable}}c
   c c@{}}
\toprule
\multirow{2}{*}{Metric} & {} & \multicolumn{2}{c}{EPFL} & \multicolumn{2}{c}{Hmito} & \multicolumn{2}{c}{Rmito} & \multicolumn{2}{c}{VNC} \\
&  &  \cellcolor{SecondaryColumnColor} & \cellcolor{SecondaryColumnColor}\textit{pval.} &  & \textit{pval.} &  \cellcolor{SecondaryColumnColor} & \cellcolor{SecondaryColumnColor}\textit{pval.} &  & \textit{pval.} \\
\cmidrule{1-10}
\multirow{3}{*}{\begin{tabular}{@{}c@{}}CTE-EI \\ \textit{Gauss} \\ \textit{(a001-003)}\end{tabular}} & K$\tau$ & 0.71 & (**) & 0.64 & (*) & 0.56 & (*) & 0.15 & (0.54) \\
& S$\rho$ & 

# DO EI (finetuned)

In [5]:
base_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/feature_perturbation_consistency/self_supervised/EI_consistency/CMB"
selected_augmentations = ["a0001", "a0005", "a001", "a002", "a003", "a004", "a005", "a01", "a015", "a02", "a025", "a03"]

DO_EI_dfs: List[pd.DataFrame] = []

for aug in selected_augmentations:
    file_name = f"transfer_DO_{aug}_CMB_05f_05b_EI_scores.json"
    consistency_path = Path(base_path) / file_name
    results = load_transfer_metric_results(consistency_path)
    consistency_scores = results["transfer_scores"]
    targets = list(consistency_scores.keys())
    consistency_scores = match_model_names(
        performance_scores=performance_scores,
        transfer_scores=consistency_scores,
    )
    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=targets,
        transfer_metric_per_target=consistency_scores,
        performance_per_target=performance_scores,
        )
    df_DO_EI = correlation_table(KT_scores, SP_scores, PE_scores, targets=targets, num_sig_fig=3)
    print(df_DO_EI)
    DO_EI_dfs.append(df_DO_EI)

Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/feature_perturbation_consistency/self_supervised/EI_consistency/CMB/transfer_DO_a0001_CMB_05f_05b_EI_scores.json
                 kt  kt pval  s rho  s rho pval     pr  pr pval
Task targets                                                   
Mito EPFL     0.564    0.018  0.727       0.016  0.910    0.000
     Hmito    0.418    0.068  0.618       0.032  0.586    0.058
     Rmito    0.600    0.010  0.791       0.006  0.810    0.002
     VNC      0.273    0.272  0.413       0.166  0.254    0.425
Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/feature_perturbation_consistency/self_supervised/EI_consistency/CMB/transfer_DO_a0005_CMB_05f_05b_EI_scores.json
                 kt  kt pval  s rho  s rho pval     pr  pr pval
Task targets                                                   
Mito EPFL     0.

In [6]:
metric_names= ['CTE-EI']*len(selected_augmentations)
names2 = ['DropOut']*len(selected_augmentations)
names3 = [ f'({aug})' for aug in selected_augmentations]

latex_table = dataframe_to_latex_table_styled(
    DO_EI_dfs, 
    metric_names, 
    names2,
    names3,
    caption="Post-UDA CTE-EI Correlation scores for Semantic Segmentation of Mitochondria across a range of DropOut Feature Perturbation Strengths. \textit{pval.} $<$ 0.05 (*), \textit{pval.} $<$ 0.01 (**).",
    label="tab:cte_ei_mitochondria_UDA_dropout"
)
print(latex_table)

\begin{table}[tb]
\centering
\scriptsize
\caption{Post-UDA CTE-EI Correlation scores for Semantic Segmentation of Mitochondria across a range of DropOut Feature Perturbation Strengths. 	extit{pval.} $<$ 0.05 (*), 	extit{pval.} $<$ 0.01 (**).}
\setlength{\tabcolsep}{3pt}
\begin{tabular}{@{}c c
   >{\columncolor{GreyTable}}c
   >{\columncolor{GreyTable}}c
   c c
   >{\columncolor{GreyTable}}c
   >{\columncolor{GreyTable}}c
   c c@{}}
\toprule
\multirow{2}{*}{Metric} & {} & \multicolumn{2}{c}{EPFL} & \multicolumn{2}{c}{Hmito} & \multicolumn{2}{c}{Rmito} & \multicolumn{2}{c}{VNC} \\
&  &  \cellcolor{SecondaryColumnColor} & \cellcolor{SecondaryColumnColor}\textit{pval.} &  & \textit{pval.} &  \cellcolor{SecondaryColumnColor} & \cellcolor{SecondaryColumnColor}\textit{pval.} &  & \textit{pval.} \\
\cmidrule{1-10}
\multirow{3}{*}{\begin{tabular}{@{}c@{}}CTE-EI \\ \textit{DropOut} \\ \textit{(a0001)}\end{tabular}} & K$\tau$ & 0.56 & (*) & 0.42 & (0.07) & 0.60 & (*) & 0.27 & (0.27) \\
& S$\rho$ 

# AdaBN

### EI (Gauss)

In [5]:
AdaBN_performance_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/transfer_AdaBN_train_performance_scores.json"
with open(AdaBN_performance_path, "r") as f:
    AdaBN_performance_scores = json.load(f)["performance_scores"]


In [6]:
base_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency/AdabN"
selected_augmentations = ["a001-a003", "a003-a005", "a005-a007", "a007-a01", "a01-a012", "a012-a015", 'a015-a02']
# selected_augmentations = ["a007-a01"]

gauss_EI_AdaBN_dfs: List[pd.DataFrame] = []

for aug in selected_augmentations:
    file_name = f"transfer_AdaBN_train_Gauss_{aug}_CMB_05f_05b_EI_scores.json"
    consistency_path = Path(base_path) / file_name
    results = load_transfer_metric_results(consistency_path)
    consistency_scores = results["transfer_scores"]
    targets = list(consistency_scores.keys())
    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=targets,
        transfer_metric_per_target=consistency_scores,
        performance_per_target=AdaBN_performance_scores,
        )
    df_gauss_AdaBN = correlation_table(KT_scores, SP_scores, PE_scores, targets=targets, num_sig_fig=4)
    print(df_gauss_AdaBN)
    gauss_EI_AdaBN_dfs.append(df_gauss_AdaBN)


Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency/AdabN/transfer_AdaBN_train_Gauss_a001-a003_CMB_05f_05b_EI_scores.json
                  kt  kt pval   s rho  s rho pval      pr  pr pval
Task targets                                                      
Mito EPFL     0.6667   0.0080  0.7972      0.0060  0.9295   0.0000
     Hmito    0.7273   0.0020  0.8462      0.0020  0.9278   0.0000
     Rmito    0.5455   0.0100  0.7133      0.0160  0.9228   0.0000
     VNC      0.3333   0.2977  0.4500      0.2537  0.5702   0.1089
Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency/AdabN/transfer_AdaBN_train_Gauss_a003-a005_CMB_05f_05b_EI_scores.json
                  kt  kt pval   s rho  s rho pval      pr  pr pval
Task targets                                                      
Mito EPFL     0.6970   0.0020  0.8671      0.0020  0.8

In [ ]:
metric_names= ['CTE-EI']*len(selected_augmentations)
names2 = ['Gauss']*len(selected_augmentations)
names3 = [ f'({aug})' for aug in selected_augmentations]

latex_table = dataframe_to_latex_table_styled(
    gauss_EI_AdaBN_dfs, 
    metric_names, 
    names2,
    names3,
    caption="Post-UDA (AdaBN) CTE-EI Correlation scores for Semantic Segmentation of Mitochondria across a range of Gaussian Input Perturbation Strengths. \textit{pval.} $<$ 0.05 (*), \textit{pval.} $<$ 0.01 (**).",
    label="tab:cte_ei_mitochondria_UDA_gauss_adabn"
)
print(latex_table)

### Transfer Score (AdaBN)

In [9]:
base_path = "/g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/1k_pixels_sampled/transfer_metric_results/AdaBN_results"
targets = ["ETarget", "HTarget", "RTarget", "VTarget"]

per_target_AdaBN_dfs = []
for target in targets:
    file_name = f"{target}_AdaBN_Transfer_Score.json"
    metric_path = Path(base_path) / file_name
    results = load_transfer_metric_results(metric_path)
    transfer_scores = results["transfer_scores"]
    performance_scores = results["performance_scores"]
    target_dataset = list(transfer_scores.keys())
    transfer_scores = match_model_names(
        performance_scores=performance_scores,
        transfer_scores=transfer_scores,
    )
    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=target_dataset,
        transfer_metric_per_target=transfer_scores,
        performance_per_target=performance_scores,
        )
    df_TS_AdaBN = correlation_table(KT_scores, SP_scores, PE_scores, targets=target_dataset, num_sig_fig=4)
    per_target_AdaBN_dfs.append(df_TS_AdaBN)


# Combine all per_target_dfs into a single dataframe with the same structure as df_DO_EI
df_finetuned_AdaBN_TS = pd.concat(per_target_AdaBN_dfs, axis=0)
print(df_finetuned_AdaBN_TS)

Loaded Transfer metric results from: /g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/1k_pixels_sampled/transfer_metric_results/AdaBN_results/ETarget_AdaBN_Transfer_Score.json
Experiment: ETarget_AdaBN_Transfer_Score
Targets: 1 (EPFL)
Source models: 9
Total transfers: 9
Loaded Transfer metric results from: /g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/1k_pixels_sampled/transfer_metric_results/AdaBN_results/HTarget_AdaBN_Transfer_Score.json
Experiment: HTarget_AdaBN_Transfer_Score
Targets: 1 (Hmito)
Source models: 9
Total transfers: 9
Loaded Transfer metric results from: /g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/1k_pixels_sampled/transfer_metric_results/AdaBN_results/RTarget_AdaBN_Transfer_Score.json
Experiment: RTarget_AdaBN_Transfer_Score
Targets: 1 (Rmito)
Source models: 9
Total transfers: 9
Loaded Transfer metric results from: /g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/1k_pixels_sampled/t

# latex Table

In [9]:
metric_names = ['CTE-EI','CTE-EI', 'TS', 'TS']
names2 = ['Gauss','Gauss-AdaBN', "MT", "AdaBN"]
correlation_dfs = [df_gauss_EI, df_gauss_AdaBN, df_finetuned_TS, df_finetuned_AdaBN_TS]
latex_table = dataframe_to_latex_table(correlation_dfs, metric_names, names2)
print(latex_table)

\begin{table*}[htbp]
\centering
\setlength{\tabcolsep}{3pt}
\begin{tabular}{cc|ccc|ccc|ccc|ccc}
\hline
\multirow{2}{*}{Metric} & {} & \multicolumn{3}{c|}{EPFL} & \multicolumn{3}{c|}{Hmito} & \multicolumn{3}{c|}{Rmito} & \multicolumn{3}{c}{VNC} \\
 &  & K$\tau$ & S$\rho$ & P$r$ & K$\tau$ & S$\rho$ & P$r$ & K$\tau$ & S$\rho$ & P$r$ & K$\tau$ & S$\rho$ & P$r$ \\
\hline
\multirow{2}{*}{\begin{tabular}{@{}c@{}}CTE-EI \\ Gauss\end{tabular}} &  & 0.78 & 0.91 & 0.93 & 0.75 & 0.83 & 0.76 & 0.64 & 0.81 & 0.70 & 0.21 & 0.24 & 0.28 \\
 & \textbf{pval.} & (0.00) & (0.00) & (0.00) & (0.00) & (0.01) & (0.01) & (0.01) & (0.01) & (0.02) & (0.37) & (0.44) & (0.38) \\
\hline
\multirow{2}{*}{\begin{tabular}{@{}c@{}}CTE-EI \\ Gauss-AdaBN\end{tabular}} &  & 0.70 & 0.86 & 0.76 & 0.73 & 0.88 & 0.91 & 0.55 & 0.73 & 0.87 & 0.39 & 0.47 & 0.71 \\
 & \textbf{pval.} & (0.01) & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) & (0.01) & (0.00) & (0.00) & (0.21) & (0.25) & (0.03) \\
\hline
\multirow{2}{*}{\begin{tabular}{@

# latex Table (Swapped Format)

In [10]:
metric_names = ['CTE-EI','CTE-EI', 'TS', 'TS']
names2 = ['Gauss','Gauss-AdaBN', "MT", "AdaBN"]
correlation_dfs = [df_gauss_EI, df_gauss_AdaBN, df_finetuned_TS, df_finetuned_AdaBN_TS]
latex_table_swapped = dataframe_to_latex_table_swapped(correlation_dfs, metric_names, names2)
print(latex_table_swapped)

\begin{table*}[htbp]
\centering
\setlength{\tabcolsep}{3pt}
\begin{tabular}{cc|cc|cc|cc|cc}
\hline
\multirow{2}{*}{Metric} & {} & \multicolumn{2}{c|}{EPFL} & \multicolumn{2}{c|}{Hmito} & \multicolumn{2}{c|}{Rmito} & \multicolumn{2}{c}{VNC} \\
 &  &  & \textbf{pval.} &  & \textbf{pval.} &  & \textbf{pval.} &  & \textbf{pval.} \\
\hline
\multirow{3}{*}{\begin{tabular}{@{}c@{}}CTE-EI \\ Gauss\end{tabular}} & K$\tau$ & 0.78 & (0.0) & 0.75 & (0.0) & 0.64 & (0.0) & 0.21 & (0.4) \\
 & S$\rho$ & 0.91 & (0.0) & 0.83 & (0.0) & 0.81 & (0.0) & 0.24 & (0.4) \\
 & P$r$ & 0.93 & (0.0) & 0.76 & (0.0) & 0.70 & (0.0) & 0.28 & (0.4) \\
\hline
\multirow{3}{*}{\begin{tabular}{@{}c@{}}CTE-EI \\ Gauss-AdaBN\end{tabular}} & K$\tau$ & 0.70 & (0.0) & 0.73 & (0.0) & 0.55 & (0.0) & 0.39 & (0.2) \\
 & S$\rho$ & 0.86 & (0.0) & 0.88 & (0.0) & 0.73 & (0.0) & 0.47 & (0.2) \\
 & P$r$ & 0.76 & (0.0) & 0.91 & (0.0) & 0.87 & (0.0) & 0.71 & (0.0) \\
\hline
\multirow{3}{*}{\begin{tabular}{@{}c@{}}TS \\ MT\end{tabular}} & K$